# Descriptive analysis of tweet sentiment data

This notebook explores the **training split** (50,306 tweets in the current dataset snapshot). It describes class balance, tweet length, common words and phrases, hashtags and mentions, and several visible text markers. It does not use the test split, so the test data remains untouched for model evaluation.

The goal is to describe patterns, not infer causes or assess classifier performance. The cleaning function is the same one used by the project models: lowercase text, replace URLs with `URL`, replace mentions with `USER`, and remove the `#` from hashtags. Raw text is retained for analyses that need the original hashtags, mentions, and URLs.

In [ ]:
from collections import Counter
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# Find the project root whether the notebook is launched from the repo or notebooks/.
starting_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (starting_directory, *starting_directory.parents)
     if (path / "src" / "tweet_sentiment").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find src/tweet_sentiment from the notebook's working directory.")
sys.path.insert(0, str(project_root / "src"))

from tweet_sentiment.data import load_train
from tweet_sentiment.preprocessing import clean_text

sns.set_theme(style="whitegrid", context="notebook")
tweets = load_train()
tweets["clean_text"] = tweets["text"].map(clean_text)
tweets["n_characters"] = tweets["text"].str.len()
tweets["n_words"] = tweets["text"].str.split().str.len()
print(f"Loaded {len(tweets):,} training tweets with columns {list(tweets.columns[:3])}.")
display(tweets.head())

## 1. Data quality and structure

Check the schema, missing values, repeated IDs, and repeated tweet texts before interpreting distributions. Duplicate rows are reported rather than removed: deduplication would change the dataset and should be a separately justified data-preparation decision. The loader also ignores optional fields after the tweet text; some training rows contain timestamps.

In [ ]:
quality = pd.DataFrame({
    "dtype": tweets[["idf", "polarity", "text"]].dtypes.astype(str),
    "missing_values": tweets[["idf", "polarity", "text"]].isna().sum(),
})
display(quality)
print(f"Rows: {len(tweets):,}; columns: {len(tweets.columns[:3])}")
print(f"Repeated IDs: {tweets['idf'].duplicated().sum():,} ({tweets['idf'].duplicated().mean():.2%})")
print(f"Repeated text values: {tweets['text'].duplicated().sum():,} ({tweets['text'].duplicated().mean():.2%})")
print("Observed polarity labels:", sorted(tweets["polarity"].unique()))

**Interpretation.** The current training split contains 50,306 rows and all three expected labels (`-1`, `0`, `1`), with no missing IDs, labels, or texts. There are 779 repeated IDs (about 1.5%) and 748 repeated text values (about 1.5%). These may be genuine repeated posts or collection artefacts; this descriptive notebook leaves them in place.

## 2. Sentiment class balance

Counts and percentages show how much data is available for each sentiment. The chart uses counts; the table includes percentages to make class imbalance easier to assess.

In [ ]:
class_summary = (
    tweets["polarity"].value_counts().sort_index().rename("tweets").to_frame()
)
class_summary["percentage"] = 100 * class_summary["tweets"] / len(tweets)
display(class_summary.round(2))

ax = sns.barplot(x=class_summary.index.astype(str), y=class_summary["tweets"], color="#4C78A8")
ax.set(title="Training tweets by sentiment", xlabel="Polarity (-1 negative, 0 neutral, +1 positive)", ylabel="Number of tweets")
for container in ax.containers:
    ax.bar_label(container, fmt="%d")
plt.tight_layout()
plt.show()

**Interpretation.** The training set is imbalanced: neutral tweets are the largest class (22,564; 44.85%), followed by positive (19,902; 39.56%), while negative tweets are substantially fewer (7,840; 15.58%). This matters when comparing class-specific counts and later when evaluating classifiers: overall accuracy alone could hide weak negative-class performance.

## 3. Tweet length by sentiment

Character and whitespace-separated word counts are summarized with mean, median, and quartiles. Boxplots help show spread and outliers. Length is computed from the original text so URL and mention replacement does not alter this comparison.

In [ ]:
length_summary = tweets.groupby("polarity")[["n_characters", "n_words"]].describe().round(1)
display(length_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=tweets, x="polarity", y="n_characters", showfliers=False, ax=axes[0])
sns.boxplot(data=tweets, x="polarity", y="n_words", showfliers=False, ax=axes[1])
axes[0].set(title="Characters per tweet", xlabel="Polarity", ylabel="Characters (outliers hidden)")
axes[1].set(title="Words per tweet", xlabel="Polarity", ylabel="Whitespace-separated words (outliers hidden)")
plt.tight_layout()
plt.show()

**Interpretation.** Average tweet lengths are similar across classes. Negative tweets are slightly longer on average (119.2 characters and 20.4 words) than neutral (116.6 characters and 19.4 words) and positive tweets (114.6 characters and 19.4 words). The small differences suggest length alone is unlikely to distinguish sentiment reliably; the distributions overlap substantially.

## 4. Frequent words and two-word phrases

Word and bigram counts are calculated on the cleaned text. To avoid a ranking dominated by grammatical stopwords and the cleaner's `URL`/`USER` placeholders, common English stopwords plus those placeholders are excluded. Results are shown as occurrences per 1,000 tweets in each class, which makes the class rates more comparable despite unequal class sizes. Counts are descriptive, not sentiment-specificity scores; frequent topic names can reflect the subjects discussed in the collection.

In [ ]:
stop_words = sorted(set(ENGLISH_STOP_WORDS) | {"url", "user"})
texts = tweets["clean_text"].tolist()
labels = tweets["polarity"].to_numpy()
frequency_tables = {}

for n in (1, 2):
    vectorizer = CountVectorizer(ngram_range=(n, n), stop_words=stop_words, min_df=5)
    counts = vectorizer.fit_transform(texts)
    terms = vectorizer.get_feature_names_out()
    for label in (-1, 0, 1):
        class_counts = counts[labels == label].sum(axis=0).A1
        top_indices = class_counts.argsort()[::-1][:10]
        n_class_tweets = (labels == label).sum()
        table = pd.DataFrame({
            "term": terms[top_indices],
            "occurrences": class_counts[top_indices].astype(int),
        })
        table["per_1k_tweets"] = 1000 * table["occurrences"] / n_class_tweets
        frequency_tables[(n, label)] = table

for n, title in ((1, "Unigrams"), (2, "Bigrams")):
    print(title)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, label in zip(axes, (-1, 0, 1)):
        table = frequency_tables[(n, label)].sort_values("per_1k_tweets")
        sns.barplot(data=table, x="per_1k_tweets", y="term", color="#72B7B2", ax=ax)
        ax.set(title=f"Polarity {label}", xlabel="Occurrences per 1,000 tweets", ylabel="")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    for label in (-1, 0, 1):
        print(f"Top {title.lower()} for polarity {label}")
        display(frequency_tables[(n, label)].round(1))

**Interpretation.** Across sentiment classes, terms such as `tomorrow`, `day`, and `going` are common, suggesting that recurring events and plans are prominent in this sample. The leading phrases differ by class (for example, `boko haram` and `charlie hebdo` appear among negative bigrams, while `tomorrow night` and entertainment-related names appear in neutral/positive lists). These are topic associations, not proof that a phrase expresses a particular sentiment. The token `u002c` also appears in frequent words, indicating encoded text artifacts that merit separate cleaning review before drawing linguistic conclusions.

## 5. Hashtags and mentions

Hashtags and mentions are extracted from raw text because the cleaner removes `@handles` and strips the `#` marker. The table reports the most common values within each class, plus the percentage of tweets in each class containing at least one hashtag or mention.

In [ ]:
tag_patterns = {
    "hashtag": re.compile(r"(?<!\w)#([A-Za-z0-9_]+)"),
    "mention": re.compile(r"(?<!\w)@([A-Za-z0-9_]+)"),
}
tag_rows = []
for label, class_tweets in tweets.groupby("polarity"):
    raw_texts = class_tweets["text"].tolist()
    for kind, pattern in tag_patterns.items():
        values = [value.lower() for text in raw_texts for value in pattern.findall(text)]
        top = Counter(values).most_common(10)
        rows_with_tag = sum(bool(pattern.search(text)) for text in raw_texts)
        tag_rows.append({
            "polarity": label,
            "kind": kind,
            "tweets_with_any_pct": 100 * rows_with_tag / len(raw_texts),
            "top_values": ", ".join(f"{value} ({count})" for value, count in top),
        })
display(pd.DataFrame(tag_rows).round({"tweets_with_any_pct": 1}))

**Interpretation.** The most frequent tags and handles vary by class and include event, news, entertainment, and public-figure references. For example, `#raw` is common in neutral and positive tweets, while mentions such as `@microsoft` and `@youtube` recur. These may reflect the dataset's topics and collection period rather than general sentiment behavior. Mentions and hashtags should be interpreted as noisy metadata: handles can be truncated or platform-specific, and a tag can have multiple meanings.

## 6. URLs, emoticons, and punctuation

For each marker, this section reports the percentage of tweets in a class that contain it at least once. These simple regular-expression indicators are transparent but approximate: URL formats vary, and the emoticon pattern does not recognize every Unicode emoji.

In [ ]:
marker_patterns = {
    "URL": re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE),
    "emoticon": re.compile(r"(?::|;|=|8)[-o^']?[)(DPp/\\|]|<3|\^_\^"),
    "exclamation": re.compile(r"!"),
    "question mark": re.compile(r"\?"),
    "ellipsis": re.compile(r"\.{3,}"),
}
marker_rows = []
for label, class_tweets in tweets.groupby("polarity"):
    for marker, pattern in marker_patterns.items():
        present = class_tweets["text"].map(lambda text: bool(pattern.search(text)))
        marker_rows.append({"polarity": label, "marker": marker, "tweets_with_marker_pct": 100 * present.mean()})
marker_summary = pd.DataFrame(marker_rows)
display(marker_summary.pivot(index="marker", columns="polarity", values="tweets_with_marker_pct").round(1))

ax = sns.barplot(data=marker_summary, x="marker", y="tweets_with_marker_pct", hue="polarity")
ax.set(title="Tweets containing each text marker", xlabel="Marker", ylabel="Tweets containing marker (%)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

**Interpretation.** URLs appear in about 32.8% of neutral tweets, 25.5% of positive tweets, and 19.7% of negative tweets. Exclamation marks are much more common in positive tweets (33.1%) than neutral (12.5%) or negative (12.8%), while question marks are somewhat less common in positive tweets. Emoticons are common in all groups (23.0%–34.6%). These are associations in this dataset and could be affected by topic, quoting, or collection practices.

## 7. Summary and limitations

- The dataset has all three sentiment labels, but the negative class is much smaller (15.58%) than neutral (44.85%) and positive (39.56%). Keep class imbalance in mind in downstream evaluation.
- Tweet length is broadly similar across labels; the mean differences are modest compared with the overall spread.
- Frequently occurring words, bigrams, hashtags, and handles show recurring topics. They should not be read as universal sentiment indicators or causal explanations.
- Visible marker rates vary by polarity in this sample (notably exclamation marks and URLs), but this is descriptive association only.
- The current text cleaner replaces URLs and mentions with `URL` and `USER`, and removes the `#` before hashtags. That is suitable for the existing model pipeline but means original values must be extracted from raw text for metadata analysis.
- Frequent `u002c` tokens suggest encoded text artifacts remain. The basic word tokenizer and regex metadata detectors also miss some language-specific forms and Unicode emoji.
- Repeated IDs/texts are reported but not removed. Any deduplication, stopword policy, or special-token normalization should be decided and documented before modeling or final reporting.

All statistics in this notebook describe the supplied training split only; they do not establish causal relationships or generalize automatically beyond this dataset.